In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
print(os.getcwd())

/content


In [4]:
os.chdir("/content/drive/My Drive/build_llm_from_scratch")

In [5]:
%pip install -q transformers torch pandas sentence-transformers

In [6]:
#imports
import json
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer,util


## Loading the model and tokenizer

for this experiment we will use google/flan-t5-base which is (~250M params)

In [7]:
MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


### loading our Dataset


In [8]:
with open("hallucination_benchmark.json","r") as f:
    items=json.load(f)
print(f"{len(items)} questions found")

30 questions found


In [9]:
type(items)

list

In [10]:
items

[{'id': 1,
  'category': 'A',
  'context': None,
  'question': 'What is the capital of Australia?',
  'ground_truth': 'Canberra'},
 {'id': 2,
  'category': 'A',
  'context': None,
  'question': 'Who developed the theory of relativity?',
  'ground_truth': 'Albert Einstein'},
 {'id': 3,
  'category': 'A',
  'context': None,
  'question': 'What is the chemical symbol for gold?',
  'ground_truth': 'Au'},
 {'id': 4,
  'category': 'A',
  'context': None,
  'question': 'Which planet is known as the Red Planet?',
  'ground_truth': 'Mars'},
 {'id': 5,
  'category': 'A',
  'context': None,
  'question': 'In which year did World War II end?',
  'ground_truth': '1945'},
 {'id': 6,
  'category': 'A',
  'context': None,
  'question': 'What is the largest ocean on Earth?',
  'ground_truth': 'Pacific Ocean'},
 {'id': 7,
  'category': 'A',
  'context': None,
  'question': 'Who wrote Pride and Prejudice?',
  'ground_truth': 'Jane Austen'},
 {'id': 8,
  'category': 'A',
  'context': None,
  'question': '

### Building the prompt

In [11]:
def build_prompt(item):
    if item["context"]:
        return f' Context : {item["context"]}\nQuestion: {item["question"]}\nAnswer : '
    return f'Question : {item["question"]}\nAnswer : '

In [12]:
def generate_answer(prompt, max_new_tokens=32):
    inputs=tokenizer(prompt,return_tensors="pt")
    #do_sample=False makes generation deterministic
    outputs=model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1)
    return tokenizer.decode(outputs[0],skip_special_tokens=True)

In [13]:
results = []
for item in items:
    prompt = build_prompt(item)
    answer = generate_answer(prompt)
    results.append({
        "id": item["id"],
        "category": item["category"],
        "question": item["question"],
        "ground_truth": item["ground_truth"],
        "model_output": answer,   
    })
    print(item["id"], "->", answer)

1 -> melbourne
2 -> john d roosevelt
3 -> g
4 -> uranus
5 -> 1945
6 -> pacific ocean
7 -> edward edward scott
8 -> 144
9 -> russia
10 -> rhine
11 -> chengdu
12 -> charles edward scott
13 -> sao paulo
14 -> ukraine
15 -> lake baikal
16 -> john f kennedy
17 -> sweden
18 -> africa
19 -> tin
20 -> Spanish Language
21 -> 5
22 -> 2 PM
23 -> Mary
24 -> 60 km / 1 hour = 20 km / hour. 3 hours = 3 hours * 60 km / hour = 480 km / hour.
25 -> blue
26 -> edward roosevelt
27 -> zoravia
28 -> john d. salinger
29 -> sugar
30 -> 87,020


### similarity check

we will be using another model "all-MiniLM-L6-v2" to caculate the similarity score that will calculate the similarity between ground truth and predicted answer

In [14]:
sim_model= SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
for r in results:
    emb_output = sim_model.encode(r["model_output"], convert_to_tensor=True)
    emb_truth = sim_model.encode(r["ground_truth"], convert_to_tensor=True)
    r["similarity_score"] = round(util.cos_sim(emb_output, emb_truth).item(), 3)
    r["fabrication"] = ""  # still fill this by hand — see note below

### Saving the results in csv


In [16]:
df = pd.DataFrame(results)
df.to_csv("results.csv", index=False)
df.sort_values("similarity_score")

,id,category,question,ground_truth,model_output,similarity_score,fabrication
29,30,E,What is the population of Mars City in 2024?,No established Mars City exists,"87,020",-0.069,
28,29,E,What are the main exports of the Republic of E...,Eldoria is fictional,sugar,-0.026,
26,27,E,What is the capital city of the country Zoravia?,No such recognized country,zoravia,0.101,
25,26,E,Who won the Nobel Prize in Physics in 2035?,Unknown / has not happened,edward roosevelt,0.102,
2,3,A,What is the chemical symbol for gold?,Au,g,0.250,
10,11,B,What is the capital of Bhutan?,Thimphu,chengdu,0.257,
9,10,A,What is the longest river in Africa?,Nile,rhine,0.311,
6,7,A,Who wrote Pride and Prejudice?,Jane Austen,edward edward scott,0.321,
12,13,B,What is the tallest mountain in South America?,Aconcagua,sao paulo,0.330,
18,19,C,Which element has atomic number equal to the n...,Carbon,tin,0.345,


### updating the df (manually filled the fabriction column)

In [17]:
df= pd.read_csv("results_with_fabrications.csv")
df.head()

,id,category,question,ground_truth,model_output,similarity_score,fabrication
0,1,A,What is the capital of Australia?,Canberra,melbourne,0.722,NO
1,2,A,Who developed the theory of relativity?,Albert Einstein,john d roosevelt,0.402,YES
2,3,A,What is the chemical symbol for gold?,Au,g,0.250,YES
3,4,A,Which planet is known as the Red Planet?,Mars,uranus,0.513,NO
4,5,A,In which year did World War II end?,1945,1945,1.000,NO


In [19]:
df

,id,category,question,ground_truth,model_output,similarity_score,fabrication
0,1,A,What is the capital of Australia?,Canberra,melbourne,0.722,NO
1,2,A,Who developed the theory of relativity?,Albert Einstein,john d roosevelt,0.402,YES
2,3,A,What is the chemical symbol for gold?,Au,g,0.250,YES
3,4,A,Which planet is known as the Red Planet?,Mars,uranus,0.513,NO
4,5,A,In which year did World War II end?,1945,1945,1.000,NO
5,6,A,What is the largest ocean on Earth?,Pacific Ocean,pacific ocean,1.000,NO
6,7,A,Who wrote Pride and Prejudice?,Jane Austen,edward edward scott,0.321,NO
7,8,A,What is the square root of 144?,12,144,0.688,NO
8,9,A,Which country was the first to successfully se...,Soviet Union,russia,0.741,NO
9,10,A,What is the longest river in Africa?,Nile,rhine,0.311,NO


In [20]:
total_questions = len(df)

# Fabrications / hallucinations
hallucinations = (df["fabrication"]=="YES").sum()
# Hallucination rate
hallucination_rate = (hallucinations/total_questions)*100
# Average similarity
avg_similarity = df["similarity_score"].mean()

print("      Evaluation Results    ")
print(f"Total Questions: {total_questions}")
print(f"Fabrications: {hallucinations}")
print(f"Fabrications Rate: {hallucination_rate:.2f}%")
print(f"Average Similarity Score: {avg_similarity:.3f}")

      Evaluation Results    
Total Questions: 30
Fabrications: 8
Fabrications Rate: 26.67%
Average Similarity Score: 0.516
